### Scrape La Liga Official Website for Results Data

In [9]:
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from bs4 import BeautifulSoup

TEAM_CLASS = 'styled__TextStyled-sc-1mby3k1-0 jTXYPl'
SCORE_CLASS = 'styled__TextStyled-sc-1mby3k1-0 gqRVUt'

In [16]:
def scrape_results_by_gameday(gameday=1):
    response = requests.get('https://www.laliga.com/en-ES/laliga-easports/results/2024-25/gameweek-%s' % str(gameday))
    soup = BeautifulSoup(response.content, 'html.parser')
    teams = soup.find_all("p", {"class": TEAM_CLASS})
    scores = soup.find_all("p", {"class": SCORE_CLASS})
    data = []
    games = []
    for i in range(10):
        try:
            home_team = teams[2 * i].text
            away_team = teams[2 * i + 1].text

            ###### Comment section to grab just teams ######
            home_score = int(scores[2 * i].text)
            away_score = int(scores[2 * i + 1].text)

            row = [home_team, away_team, home_score, away_score]
            data.append(row)

            ################################################
            games.append((home_team, away_team))
        except:
            break
    data = pd.DataFrame(data, columns=['home_team', 'away_team', 'home_score', 'away_score'])
    
    return data
        
def scrape_all_results(from_gameday, to_gameday):
    res = pd.DataFrame()
    for gameday in tqdm(range(from_gameday, to_gameday + 1)):
        gameday_df = scrape_results_by_gameday(gameday)
        gameday_df['gameday'] = gameday
        res = res.append(gameday_df)
    return res

In [17]:
df = scrape_all_results(1, 18)

  0%|                                                    | 0/18 [00:00<?, ?it/s]/var/folders/kq/1x0wvztd0r5cgtjbqg2gvr6w0000gn/T/ipykernel_27560/3504290937.py:33: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  res = res.append(gameday_df)
  6%|██▍                                         | 1/18 [00:01<00:23,  1.36s/it]/var/folders/kq/1x0wvztd0r5cgtjbqg2gvr6w0000gn/T/ipykernel_27560/3504290937.py:33: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  res = res.append(gameday_df)
 11%|████▉                                       | 2/18 [00:02<00:18,  1.17s/it]/var/folders/kq/1x0wvztd0r5cgtjbqg2gvr6w0000gn/T/ipykernel_27560/3504290937.py:33: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  res = res.append(gameday_df)
 17%|███████▎      

In [18]:
df

,home_team,away_team,home_score,away_score,gameday
0,Athletic Club,Getafe CF,1,1,1
1,Real Betis,Girona FC,1,1,1
2,RC Celta,Deportivo Alavés,2,1,1
3,UD Las Palmas,Sevilla FC,2,2,1
4,CA Osasuna,CD Leganés,1,1,1
...,...,...,...,...,...
5,Valencia CF,Deportivo Alavés,2,2,18
6,Real Madrid,Sevilla FC,4,2,18
7,CD Leganés,Villarreal CF,2,5,18
8,UD Las Palmas,RCD Espanyol de Barcelona,1,0,18


In [19]:
df.to_csv('laliga_results_raw_2425.csv')

In [24]:
df = pd.read_csv('laliga_results_raw.csv')

In [25]:
df['num_goals'] = df['home_score'] + df['away_score']

In [285]:
def construct_matrix(df, col='num_goals'):
    teams = sorted(df.home_team.unique().tolist())
    scores_df = pd.DataFrame(data=np.empty((20, 20,)), columns=teams)
    scores_df[:] = np.nan
    scores_df.index = teams
    for i in range(len(df)):
        row = df.iloc[i]
        scores_df.loc[row['home_team'], row['away_team']] = row[col] + 0.00001
    return scores_df

In [286]:
total_score_df = construct_matrix(df, col='num_goals')
home_score_df = construct_matrix(df, col='home_score')
away_score_df = construct_matrix(df, col='away_score')

In [290]:
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error
from copy import copy

def matrix_factorization_rank_calculation(matrix_df):
    filled_df = copy(matrix_df).fillna(matrix_df.mean())
    best_RMSE = 10000
    best_rank = -1
    for i in range(19):
        rank = i + 1
        model = NMF(n_components=rank, init='random', random_state=1512, max_iter=2000)
        W = model.fit_transform(filled_df)
        H = model.components_
        V = W @ H
        RMSE = np.sqrt(mean_squared_error(filled_df, V))
        if RMSE < best_RMSE:
            best_rank = rank
        print("Rank = %s, RMSE = %s." % (rank, RMSE))

    return best_rank

In [291]:
matrix_factorization_rank_calculation(home_score_df)

Rank = 1, RMSE = 0.967151738530829.
Rank = 2, RMSE = 0.8551672950277147.
Rank = 3, RMSE = 0.7670542624121537.
Rank = 4, RMSE = 0.6797478102727309.
Rank = 5, RMSE = 0.6133430335502507.
Rank = 6, RMSE = 0.5539601145540975.
Rank = 7, RMSE = 0.4996187869273635.
Rank = 8, RMSE = 0.45352100333285045.
Rank = 9, RMSE = 0.40986438294074484.
Rank = 10, RMSE = 0.37051745098988315.
Rank = 11, RMSE = 0.3266159315806658.
Rank = 12, RMSE = 0.2904902322993279.
Rank = 13, RMSE = 0.24910808730623338.
Rank = 14, RMSE = 0.21099633537729995.
Rank = 15, RMSE = 0.18815166618839962.
Rank = 16, RMSE = 0.14053325629848706.
Rank = 17, RMSE = 0.12288144832446816.
Rank = 18, RMSE = 0.09328468986782505.
Rank = 19, RMSE = 0.06854543423546226.


/usr/local/lib/python3.9/site-packages/sklearn/decomposition/_nmf.py:1692: ConvergenceWarning: Maximum number of iterations 2000 reached. Increase it to improve convergence.
  warnings.warn(


19

In [293]:
import random
def predict(matrix_df, is_home_df=True):
    # Fill missing values with mean before doing the factorization
    filled_df = copy(matrix_df).fillna(matrix_df.mean(axis=0 if is_home_df else 1))

    model = NMF(n_components=8, init='random', random_state=random.randint(1, 1000), max_iter=1000)
    W = model.fit_transform(filled_df)
    H = model.components_
    V = W @ H
    res_df = pd.DataFrame(data=V, columns=filled_df.columns, index=filled_df.index)
    return V

In [298]:
from collections import defaultdict

def pred_next_gameday(gameday=33, simulation_times=100):
    response = requests.get('https://www.laliga.com/en-GB/laliga-easports/results/2023-24/gameweek-%s' % str(gameday))
    soup = BeautifulSoup(response.content, 'html.parser')
    teams = soup.find_all("p", {"class": TEAM_CLASS})
    
    prediction_dict = defaultdict(lambda: defaultdict(int))
    
    for i in tqdm(range(simulation_times)):
        complete_home_score_df = pd.DataFrame(data=np.zeros((20, 20)), columns=home_score_df.columns, index=home_score_df.index)
        complete_away_score_df = pd.DataFrame(data=np.zeros((20, 20)), columns=away_score_df.columns, index=away_score_df.index)
        complete_home_score_df += predict(home_score_df, is_home_df=True)
        complete_away_score_df += predict(away_score_df, is_home_df=False)
        for i in range(10):
            home_team = teams[2 * i].text
            away_team = teams[2 * i + 1].text
            pred_home_score = round(complete_home_score_df.loc[home_team, away_team])
            pred_away_score = round(complete_away_score_df.loc[home_team, away_team])
            prediction_dict['%s vs %s' % (home_team, away_team)]['%s - %s' % (pred_home_score, pred_away_score)] += 1
    return prediction_dict

In [299]:
prediction_dict = pred_next_gameday(gameday=33, simulation_times=1000)

  0%|                                                  | 0/1000 [00:00<?, ?it/s]/usr/local/lib/python3.9/site-packages/sklearn/decomposition/_nmf.py:1692: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(
  3%|█                                        | 26/1000 [00:01<00:38, 25.35it/s]/usr/local/lib/python3.9/site-packages/sklearn/decomposition/_nmf.py:1692: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(
  4%|█▌                                       | 38/1000 [00:01<00:40, 23.77it/s]/usr/local/lib/python3.9/site-packages/sklearn/decomposition/_nmf.py:1692: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(
  8%|███▏                                     | 77/1000 [00:03<00:45, 20.24it/s]/usr/local/lib/python3.9/site-packages/sklearn/decomposition/_nmf.py:1692: ConvergenceWarning: Maximum numb

In [300]:
for game in prediction_dict:
    print("Simulation for game: %s" % game)
    res_list = sorted([(score, count) for (score, count) in prediction_dict[game].items()], key=lambda x: -x[1])
    for score, count in res_list:
        print("\tScore %s appeared %s times." % (score, count))

Simulation for game: Real Sociedad vs Real Madrid
	Score 1 - 1 appeared 1000 times.
Simulation for game: UD Las Palmas vs Girona FC
	Score 2 - 1 appeared 991 times.
	Score 1 - 1 appeared 9 times.
Simulation for game: UD Almería vs Getafe CF
	Score 2 - 2 appeared 1000 times.
Simulation for game: Deportivo Alavés vs RC Celta
	Score 1 - 1 appeared 852 times.
	Score 2 - 1 appeared 148 times.
Simulation for game: Atlético de Madrid vs Athletic Club
	Score 1 - 1 appeared 1000 times.
Simulation for game: Cádiz CF vs RCD Mallorca
	Score 1 - 1 appeared 992 times.
	Score 2 - 1 appeared 8 times.
Simulation for game: Granada CF vs CA Osasuna
	Score 2 - 1 appeared 978 times.
	Score 3 - 1 appeared 22 times.
Simulation for game: Villarreal CF vs Rayo Vallecano
	Score 0 - 1 appeared 862 times.
	Score 1 - 1 appeared 74 times.
	Score 0 - 2 appeared 58 times.
	Score 1 - 2 appeared 6 times.
Simulation for game: Real Betis vs Sevilla FC
	Score 1 - 1 appeared 1000 times.
Simulation for game: FC Barcelona vs